# 🔬 Lab 04 — Malware Detection from API Call Sequences
**Home Assignment — Models for AI, Politecnico di Torino**

In this lab you will tackle a **cybersecurity** problem: detecting whether a Windows executable is malicious by analysing the sequence of **system API calls** it makes at runtime.

Malware often exhibits characteristic behavioural patterns in the API calls it invokes (e.g., loading DLLs, writing to foreign process memory, opening network sockets). 

Your task is to **build classifiers that distinguish malware from benign software** based solely on these sequences.

**How to use this notebook:**
- Red text marks your tasks - complete each cell before moving on.
- You may (and should) refer to the three previous labs.
- You are provided with a `train.json` and a `test.json` files.

## 🕵🏻 Valutazione 

- **Quando**: L’esercitazione dovrà essere consegnata tramite il portale della didattica entro domenica 5 aprile 2026. 
 
- **Come**: Le risposte dovranno essere inserite direttamente nel notebook Jupyter in cui sono presenti le domande, riportando sia il codice sia le risposte e i risultati (se necessari). Il file dovrà essere esportato e consegnato esclusivamente in formato HTML, nominato nel seguente modo: `<matricola>.html`
 
- **Valutazione**: La valutazione terrà conto della correttezza della metodologia applicata, della chiarezza e coerenza delle risposte fornite, nonché dell’eventuale adeguata giustificazione delle scelte effettuate.

## Objectives of the lab

To complete the lab assignment, the student will:

1. **Characterise** a dataset of categorical sequence data and reason about its challenges (class imbalance, vocabulary coverage, variable lengths).
2. **Build a Shallow Learning Baseline** with a Random Forest — connecting to Lab 01.
3. **Use learned embeddings** to encode categorical tokens for a feedforward model — connecting to Lab 02.
4. **Build an LSTM classifier** that processes embedded categorical sequences — combining embeddings (Lab 02) with sequential processing (Lab 03).
5. **Compare models** and reason about why simpler approaches can outperform more complex ones on a given task.

In [ ]:
############################################# DO NOT CHANGE
def turn_red(text):
    return f"\x1b[1;31m{text}\x1b[0m"

from IPython.display import Image, display
def display_image(path, width=None, height=None):
    display(Image(path, width=width, height=height))
    
RANDOM_STATE = 42

In [ ]:
print(turn_red("Fix the seed for reproducibiliy!"))

############################################# YOUR SOLUTION HERE

---
## Block 01 — Data Loading & Characterisation

Each sample in the dataset is a **process execution trace** collected inside a dynamic malware analysis sandbox. It contains two fields:

- `api_call_sequence`: a list of Windows API call names (strings), in the order
  they were invoked during execution.
- `is_malware`: binary label - `1` if malicious, `0` if benign.

Before building any model, take the time to understand the data. This step will directly inform your modelling choices.

In [ ]:
print(turn_red("Q) Load the training and test datasets. How many samples does each split contain?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Inspect the first training sample. What fields does it have?"))
print(turn_red("   What does the sequence look like? How long is it?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Compute the distribution of sequence lengths across both splits."))
print(turn_red("   Report min, max, mean, and median. Also, on the same figure, plot the training and testing length distributions using histograms."))
print(turn_red("   Compared to Lab 03 (CharacterTrajectories: 60–182 steps), how variable are these lengths?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Compute the class distribution in both splits. How imbalanced is the dataset?"))
print(turn_red("   What does this imply for:"))  
print(turn_red("      (a) your choice of loss function? Do you think that it is safe to use an unweighted one?"))
print(turn_red("      (b) which evaluation metric to trust? Is `accuracy` a reliable indicator?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) How many unique API calls appear in the training set? In the test set?"))
print(turn_red("   Are there calls in the test set that never appeared in training (OOV)?"))
print(turn_red("   Why does this matter, and how will you handle it?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Which API calls are most characteristic of malware? Of benign samples? Are the top-10 calls the same?"))
print(turn_red("   Plot the top-10 most 'malware-biased' and 'benign-biased' calls."))
print(turn_red("   Remember: normalize counters with respect to the number of benign and malware calls."))
print(turn_red("   Notice: to compute the number of times a malware/benign call is made:"))
print(turn_red("       - Initialize two dictionaries - one counting the frequency of malware calls, the other of benign calls"))
print(turn_red("       - Iterate over benign and malware sequences"))
print(turn_red("       - For each sequence, consider all calls in the sequence"))
print(turn_red("       - Update the call counter"))
print(turn_red("   Eventually, normalize the number of calls according to the number of benign and malign calls."))
print(turn_red("   Sort the dictionaries to retrieve the the top-10 calls. Use `sorted(your_dict.items(), key: lambda x:x[1], reverse=True)` to do so."))

############################################# YOUR SOLUTION HERE

---
## Block 02 — Bag-of-Words Baseline

The simplest way to represent a sequence of words is by using the so-called **Bag of Words (BoW)** approach.

In BoW approaches, each sequence is a row and the vocabulary (i.e., all words that can be used in the sequences) columns.

To build a BoW matrix, we count how many times each word appears in the sequence. We put 0 if the word never happears in that sequence. 

Observe the following schema, in which we provide an example given two sequences of calls:

In [ ]:
display_image("schema/BoW.png", width=600, height=300)

**In practise**: each sample becomes a histogram over the vocabulary — a fixed-size vector that any classic ML model can consume.

This directly connects to **Lab 01**: you will use a **Random Forest** classifier, tuned with cross-validated grid search.

In [ ]:
print(turn_red("Q) With this approach, are we keeping the original order of the calls?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("In the following, we provide a function to implement build the BoW matrix starting from some input sequences."))
print(turn_red("Notice: The method requires a vocabulary - all the api calls in the dataset, that you will use as columns to construct the matrix."))
print(turn_red("Q) Which partition do you use to extract the vocabulary? Can you use the test set?"))

############################################# DO NOT CHANGE
def build_bow_features(sequences, vocab):
    """
    Build a Bag-of-Words count matrix.
      - sequences : list of lists of strings
      - vocab     : set/list of known tokens (built from the TRAINING set)
    Returns a float32 numpy array of shape (n_samples, |vocab| + 1).
    The last column accumulates counts for unknown tokens (UNK).
    """
    vocab_to_idx = {w: i for i, w in enumerate(sorted(vocab))}
    X = np.zeros((len(sequences), len(vocab) + 1), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for call in seq:
            idx = vocab_to_idx.get(call, len(vocab))   # unknown → UNK column
            X[i, idx] += 1
    return X

In [ ]:
print(turn_red("Q) Use build_bow_features() to build BoW matrices for train and test."))
print(turn_red("   What shape do the resulting matrices have? Is it consistent with previous results?"))
print(turn_red("   How sparse are they on average (fraction of non-zero columns per row)?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Train a Shallow Learning Classifier with cross-validated hyperparameter search (recall Lab 01)."))
print(turn_red("   Use class_weight='balanced'. Optimise for F1."))
print(turn_red("   Why is accuracy a poor scoring metric here?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Evaluate the best RF on the test set."))
print(turn_red("   Report the full classification report and the confusion matrix."))

############################################# YOUR SOLUTION HERE

---
## Block 03 — FFNN with Learned Embeddings

BoW approaches represent simple baselines that gives away some info of the original data (e.g., the order).

Following the example of Lab 2, we can do something more sophisticated than that. 

For instance, rather than representing each API call with a raw count, we can learn a **dense embedding** — a compact vector that the network adjusts during training so that semantically similar calls end up close together in embedding space.

We will try this approach initially relying on FFNN.

In [ ]:
print(turn_red("Q) Do you think that FFNN are the most suitable choice here? Why?"))
print(turn_red("Q) How do you think you will have to deal with the problem? How did we tackle this scenario in Lab 03?"))

In [ ]:
print(turn_red("In the following, we provide two fuctions (`build_vocab` and `encode_sequences`)."))
print(turn_red("Use these function to encode the input string sequences into numerical identifiers that the FFNN can process."))
print(turn_red("Q) We are introducing 2 extra words, <PAD> and <UNK>. Why do you think it is the case?"))
print(turn_red("Q) Which partition do you use to build the vocabulary?"))
print(turn_red("Q) How many words are there in the vocabulary? Is it consistent with previous results?"))
print(turn_red("Q) Print the first training sequence BEFORE and AFTER encoding."))

############################################# DO NOT CHANGE

def build_vocab(sequences):
    """Map each unique token to an integer. Index 0: <PAD>, 1: <UNK>."""
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for seq in sequences:
        for token in seq:
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

def encode_sequences(sequences, vocab):
    """Replace each token with its vocab index; unknowns → <UNK> index."""
    unk = vocab['<UNK>']
    return [[vocab.get(tok, unk) for tok in seq] for seq in sequences]

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Now, build a function that i) pads or truncates sequences and ii) turns them into numpy arrays. Choose a suitable `FIXED_LEN` parameter."))
print(turn_red("Q) What is the final shape of the training dataset? What is the one of the testing dataset?"))

############################################# YOUR SOLUTION HERE


In [ ]:
print(turn_red("Q) Split the training data into 80/20. Do you need to use stratified sampling?"))

############################################# YOUR SOLUTION HERE
)

In [ ]:
print(turn_red("In the following, we provide the code to compute the class weights for each class. They will be useful later to compute the losses."))
print(turn_red("Q) What is the weight for the benign class (cw[0])? What is the weigth of the malign one (cw[1])?"))

############################################# DO NOT CHANGE

from sklearn.utils import compute_class_weight
classes_ = np.unique(y_tr)
cw = compute_class_weight('balanced', classes=classes_, y=y_tr)

############################################# YOUR SOLUTION HERE

In [ ]:
import torch
print(turn_red("Eventually, create train, validation and test dataloaders. We provide a simple function to create the datasets."))

############################################# DO NOT CHANGE

from torch.utils.data import TensorDataset, DataLoader
def create_ds(X, y):
    return TensorDataset(torch.tensor(X, dtype=torch.long),
                         torch.tensor(y, dtype=torch.long))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Now, focus on the FFNN model. This time, you are dealing with sequences of categorical features which you will have to embed (mix of Lab 2 and Lab 3)."))
print(turn_red("   - The model receives a batch B of sequences."))
print(turn_red("   - Each sequence have T API calls (you set T to `Fixed Len` earlier on)."))
print(turn_red("   - You encoded each call into a numerical identifier (`encode_sequences`)."))
print(turn_red("   - Final shape of the input data: BxTx1"))
print(turn_red("This FFNN model must:"))
print(turn_red("   - Use an embedding layer to map the identifiers into embeddings (BxTx1 > BxTxh)"))
print(turn_red("   - Similarly to Lab 2, map the 3D input matrix (BxTxh) into a 2D input matrix (Bx(Txh))"))
print(turn_red("   - While in Lab 2 we flatten the matrix, here we will aggregate the last dimension with a mean aggregator"))
print(turn_red("      In other terms, we will compute the mean over the last dimension."))
print()
print(turn_red("We provide the code to instantiate the model. Answer to the following questions:"))
print(turn_red("   - Q) How many layers does the MLP layer have?"))
print(turn_red("   - Q) Which is the line converting the numerical identifiers into embeddings?"))
print(turn_red("   - Q) Why are we using a padding mask?"))
print(turn_red("   - Q) Can we still track the orderd of the calls after the aggregation?"))


############################################# DO NOT CHANGE
from torch import nn
class EmbedderMLP(nn.Module):
    """
    MLP classifier with an embedding front-end
    Forward pass:
      1. Look up each integer token in the embedding table.
            (batch, seq_len)  →  (batch, seq_len, embedding_dim)
      2. Mean-pool the embeddings, masking out <PAD> positions 
            (batch, seq_len, embedding_dim)  →  (batch, embedding_dim)
      3. MLP head → class logits.
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_classes, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.fc1     = nn.Linear(embedding_dim, hidden_dim)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2     = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb  = self.embedding(x)                                  # (B, T, D)
        mask = (x != self.embedding.padding_idx).unsqueeze(-1)    # (B, T, 1)
        # Masked mean: sum over real tokens, divide by number of real tokens
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp(min=1)   # (B, D)
        return self.fc2(self.dropout(self.relu(self.fc1(pooled))))
  
############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("We provide some of the functions to train your models (both the FFNN and the LSTM later)."))
print(turn_red("   Q) The `run_training` function is slightly different from usual. Which is the model we are saving at the end of the loop?"))


def run_training(model, train_loader, val_loader, optimizer, criterion,
                 epochs, device, model_type='mlp', print_every=5):
    """Train with early stopping on validation loss (keeps best model)."""
    best_val_loss, best_state = float('inf'), None
    history = {'train_loss': [], 'val_loss': []}
    for epoch in range(epochs):
        tl = train_epoch(model, train_loader, optimizer, criterion, device, model_type)
        vl = eval_epoch(model, val_loader, criterion, device, model_type)
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % print_every == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:3d}/{epochs}]  Train: {tl:.4f}  Val: {vl:.4f}")
    model.load_state_dict(best_state)
    return model, history

def _forward(model, batch, device, model_type):
    """Single forward pass. Handles both MLP (2-tuple batch) and LSTM (3-tuple)."""
    if model_type == 'lstm':
        seqs, lengths, labels = batch
        seqs, labels = seqs.to(device), labels.to(device)
        logits = model(seqs, lengths)
    else:
        seqs, labels = batch
        seqs, labels = seqs.to(device), labels.to(device)
        logits = model(seqs)
    return logits, labels

def get_predictions(model, loader, device, model_type='mlp'):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            logits, labels = _forward(model, batch, device, model_type)
            preds_all.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    return np.array(preds_all), np.array(labels_all)

def train_epoch(model, loader, optimizer, criterion, device, model_type='mlp'):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        logits, labels = _forward(model, batch, device, model_type)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, criterion, device, model_type='mlp'):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            logits, labels = _forward(model, batch, device, model_type)
            total_loss += criterion(logits, labels).item()
            preds_all.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    return avg_loss

In [ ]:
print(turn_red("Q) Instantiate and train an EmbedderMLP. Choose hyperparameters (hidden_dim, lr, epochs, etc.) and the optimizer."))
print(turn_red("Q) Consider the number of API calls in your dataset. Which is a good value for the embedding_dim to not compress too much the information?"))
print(turn_red("   Notice: we already provided the code for the weighted CrossEntropyLoss."))
print(turn_red("Q) Plot the training and validation lossess. use the `history` output of `run_training`."))

############################################# DO NOT CHANGE
import random
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed(RANDOM_STATE)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Use weights we previously obtained
class_weights_emb = torch.tensor(cw, dtype=torch.float32).to(device)
# Create a weighted crossentropy loss
criterion_emb = nn.CrossEntropyLoss(weight=class_weights_emb)

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Evaluate the EmbedderMLP on the test set."))
print(turn_red("   Q) How does it compare to the BoW baseline? Why do you think it is the case?"))

############################################# YOUR SOLUTION HERE

---
## Block 04 — LSTM Classifier on Categorical Sequences

Both BoW and the EmbedderMLP ignore the **order** of API calls. 

Contrarily, an LSTM can, in principle, capture temporal patterns 
   e.g., *"LdrLoadDll followed immediately by NtWriteVirtualMemory is more suspicious than either call alone"*.

In the following, you will combine what you learned in Lab 02 and Lab 03. Specifically, you will use:

- **Embedding layer** (Lab 02) → encode each categorical token as a dense vector.
- **LSTM** (Lab 03) → process the sequence of embedded vectors step by step.

The key insight is that the LSTM's `input_size` is no longer the number of raw input features - it is the **embedding dimension**: 
each timestep feeds the embedding vector for that API call into the LSTM cell.

```
integer indices  ──►  [Embedding]  ──►  dense vector per timestep
                                                    │
                                                    ▼
                                               [LSTM]  ──►  hidden state (last step)
                                                                        │
                                                                        ▼
                                                               [Linear]  ──►  logits
```

In [ ]:
print(turn_red("Create a Pytorch dataset to process the input sequences."))
print(turn_red("   Remember that you already encoded the strings into numerical identifiers: hence, you can follow the example of Lab 3 here."))
############################################# YOUR SOLUTION HERE

### Dynamic padding with a custom collate function

Sequences in a batch may have different lengths. As in Lab 03, you need a
**collate function** that:
1. Pads the sequences in a batch to the length of the longest one in that batch.
2. Records the **original lengths** so the LSTM can skip padding tokens via
   `pack_padded_sequence`.

The collate function should return a 3-tuple: `(padded_sequences, lengths, labels)`.

In [ ]:
print(turn_red("Q) Implement the collate function for the DataLoader (recall Lab 03)."))
print(turn_red("   It receives a list of (sequence_tensor, label) pairs and must return:"))
print(turn_red("     (padded_sequences, lengths, labels)"))
print(turn_red("   Pad to the longest sequence in the batch (padding_value=0)."))
print(turn_red("   Hint: as in Lab 03, you can use torch.nn.utils.rnn.pad_sequence."))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Re-use the encoded sequences from Block 03 (train_indexed, test_indexed)."))
print(turn_red("   Create an 80/20 stratified train/val split."))
print(turn_red("   Build APICallDataset objects and DataLoaders with collate_fn."))
print(turn_red("   Compute class weights from the training split."))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Use the provided `LSTMClassifier` to classify the sequences of calls."))
print(turn_red("Q) Is the embedding layer different with respect to the one of the FFNN?"))
print(turn_red("Q) Where do you iteratively process the embeddings?1"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Instantiate and train the LSTMClassifier."))
print(turn_red("   Choose hyperparameters for embedding_dim, hidden_size, n_layers, lr, epochs."))
print(turn_red("   Use a weighted CrossEntropyLoss. Plot the training history."))
print(turn_red("   Does the LSTM converge smoothly?"))

############################################# YOUR SOLUTION HERE

In [ ]:
print(turn_red("Q) Evaluate the LSTM on the test set. Report the full classification report."))
print(turn_red("   Also store predictions as y_pred_lstm for the comparison in Block 05."))

############################################# YOUR SOLUTION HERE

---
## Block 05 — Comparison & Reflection

You have now trained three very different models on the same data:

| Model        | Key idea                             | Discards order? |
|--------------|--------------------------------------|-----------------|
| BoW + RF     | Count-based features, classic ML     | Yes             |
| EmbedderMLP  | Learned token embeddings, mean-pool  | Yes             |
| LSTM         | Learned embeddings + sequential LSTM | No              |

## (OPTIONAL) Reflection Questions

Answer the following questions (as comments in the cell below or in a new
markdown cell):

1. **BoW strength.** BoW discards all order information yet achieves the highest F1. Why might counting API calls — without caring about order — be sufficient for this task? 
What would need to change in the data (dataset size? sequence length? attack complexity?) for the LSTM to outperform BoW?

2. **Evaluation metric.** Accuracy is above 90% for every model, yet F1 scores are much lower. Why?
Given the class imbalance, which metric matters more to a security analyst — precision or recall? Justify your answer in a practical context (what is the cost of a false negative here?).

3. **OOV handling.** There are 3 API calls in the test set that never appeared in training. How does each of your three models deal with them? Which approach is most robust to this kind of distribution shift?